# Antibody form converter · 1.02
**First use:** click ▶ below once to start. Then use **Choose files** to select one or several v4.2 Excel files. Conversion runs automatically.
Download each result as a ZIP containing the workbook and review report.


In [ ]:
# @title ▶ Start converter { display-mode: "form" }
import subprocess, sys
from IPython.display import clear_output
print('Preparing converter…')
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'openpyxl>=3.1.5,<4', 'lxml>=5,<7', 'ipywidgets>=7,<9'], check=True, capture_output=True, text=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError('Setup failed. Please reconnect the runtime and try again. ' + exc.stderr[-1500:]) from exc
import base64, tempfile
from pathlib import Path
_template_bytes = base64.b64decode('...')
_module_source = '"""v4.2 to v3.1 converter: retain template XML rather than rebuilding Excel styling."""\nfrom pathlib import Path\nfrom copy import deepcopy\nimport argparse, csv, difflib, io, json, re, zipfile\nfrom lxml import etree as ET\nfrom openpyxl import load_workbook\nfrom openpyxl.utils import get_column_letter\n\nVERSION = \'1.02\'\nTEMPLATE = Path(__file__).parent / \'templates/v3.1.xlsx\'\nNS = \'http://schemas.openxmlformats.org/spreadsheetml/2006/main\'\nQ = lambda name: f\'{{{NS}}}{name}\'\nMAP = {\'A\':\'A\', \'B\':\'B\', \'C\':\'C\', **{get_column_letter(i):get_column_letter(i-3) for i in range(7,23)}}\nGROUPS = {\'D\':\'Isotype\',\'E\':\'Light Chain (λ or κ)\',\'G\':\'VH Vector Name - Insertion Site Name\',\'H\':\'Kozak Setting\',\'I\':\'Kozak Sequence\',\'J\':\'Leader Setting\',\'K\':\'Leader Sequence\',\'N\':\'VL Vector Name - Insertion Site Name\',\'O\':\'Kozak Setting\',\'P\':\'Kozak Sequence\',\'Q\':\'Leader Setting\',\'R\':\'Leader Sequence\'}\nORDER_CATEGORIES = {3:\'Scale*\',4:\'Pooling\',5:\'Cell Line\',6:\'Purification\',7:\'Elution Buffer*\',8:\'Buffer Exchange\',9:\'Normatlization\',10:\'Post Purification Sterile Filtration\',11:\'Aliquots\',12:\'Shipping Container*\',13:\'Custom Plate Map\',14:\'Restriction Enzyme Sites to Avoid for DNA Optimization\',15:\'Please indicate if you prefer not to receive physical samples and only data\',16:\'Ship with testing pending\'}\nALIASES = {3:[\'scale\',\'production scale\',\'expression scale\'],4:[\'pooling\'],5:[\'cell line\',\'expression host\'],6:[\'purification\',\'purification method\'],7:[\'elution buffer\'],8:[\'buffer exchange\'],9:[\'normalization\',\'normatlization\'],10:[\'post purification sterile filtration\',\'sterile filtration\'],11:[\'aliquoting\',\'aliquots\'],12:[\'shipping container\',\'shipping format\'],13:[\'custom plate map\'],14:[\'restriction enzyme sites to avoid for dna optimization\',\'restriction enzyme sites to avoid for codon optimization\'],15:[\'please indicate if you prefer not to receive physical samples and only data\',\'please indicate if you prefer not to receive any physical samples and only data\'],16:[\'ship with testing pending\',\'ship physical samples with additional service requests pending\'],17:[\'additional requests\']}\n\nALIASES[9].append(\'Normalization (select Scale for options) Please note: we do not normalize or invoice for samples below target concentration\')\nALIASES[11].append(\'Aliquoting (Not applicable when characterization services and testing are added, select Scale for options)\')\nALIASES[12].append(\'Shipping Container* (select Scale for options)\')\n\ndef occupied(v):\n    return v is not None and str(v).strip() != \'\'\n\ndef norm(v):\n    return re.sub(r\'[^a-z0-9]+\', \'\', str(v).lower().replace(\'λ\',\'lambda\').replace(\'κ\',\'kappa\'))\n\ndef options():\n    w = load_workbook(TEMPLATE, read_only=True, data_only=True)\n    out = {}; category = None\n    for a,b in w[\'Sheet3\'].iter_rows(max_col=2, values_only=True):\n        if a: category = a\n        if category and b: out.setdefault(category, []).append(b)\n    w.close(); out[\'Cell Line\']=[\'HEK293\',\'CHO\']\n    return out\n\ndef closest(value, choices):\n    """Return actual template spelling; expose nonexact choices for review."""\n    clean = re.sub(r\'^Empty Vector\\s*-\\s*\',\'\',str(value),flags=re.I)\n    choices = [c for c in choices if not c.startswith(\'PLEASE SELECT\')]\n    for c in choices:\n        if norm(clean) == norm(c): return c, \'matched\'\n    ranked = sorted(((difflib.SequenceMatcher(None,norm(clean),norm(c)).ratio(),c) for c in choices),reverse=True)\n    if ranked and ranked[0][0] >= .55:\n        return ranked[0][1], \'closest option; review\'\n    return None, \'no reliable option; original preserved in report\'\n\ndef set_cell(root, address, value):\n    data=root.find(Q(\'sheetData\')); rn=re.search(r\'\\d+\',address).group()\n    row=data.find(f"{Q(\'row\')}[@r=\'{rn}\']")\n    if row is None:\n        row=ET.Element(Q(\'row\'),r=rn)\n        following=next((r for r in data if int(r.get(\'r\'))>int(rn)),None)\n        if following is None: data.append(row)\n        else: data.insert(data.index(following),row)\n    cell=row.find(f"{Q(\'c\')}[@r=\'{address}\']")\n    if cell is None:\n        cell=ET.SubElement(row,Q(\'c\'),r=address)\n        # Copy template input styling from row 4 for added sequence rows.\n        model=data.find(f"{Q(\'row\')}[@r=\'4\']/{Q(\'c\')}[@r=\'{re.sub(r\'[0-9]\', \'\',address)}4\']")\n        if model is not None and model.get(\'s\'): cell.set(\'s\',model.get(\'s\'))\n    for child in list(cell): cell.remove(child)\n    cell.attrib.pop(\'t\',None)\n    if occupied(value):\n        cell.set(\'t\',\'inlineStr\'); text=ET.SubElement(ET.SubElement(cell,Q(\'is\')),Q(\'t\'))\n        text.set(\'{http://www.w3.org/XML/1998/namespace}space\',\'preserve\'); text.text=str(value)\n    # Excel requires cells in column order.\n    def col(c):\n        n=0\n        for ch in re.sub(r\'\\d\',\'\',c.get(\'r\')): n=n*26+ord(ch)-64\n        return n\n    row[:]=sorted(row,key=col)\n\ndef convert(blob):\n    opts=options(); issues=[]; count=0\n    w=load_workbook(io.BytesIO(blob),read_only=True,data_only=True)\n    if \'Antibody Designs\' not in w.sheetnames:\n        w.close(); raise ValueError(\'Missing Antibody Designs tab; expected a v4.2 workbook\')\n    z=zipfile.ZipFile(TEMPLATE); roots={n:ET.fromstring(z.read(n)) for n in z.namelist() if n.startswith(\'xl/worksheets/\') and n.endswith(\'.xml\')}\n    seq=roots[\'xl/worksheets/sheet3.xml\']; details=roots[\'xl/worksheets/sheet4.xml\']\n    def report(location,original,result,status):\n        issues.append({\'location\':location,\'original\':original,\'selected\':result,\'status\':status,\'version\':VERSION})\n    # Clear template placeholders, preserving every style and instructional row.\n    for c in list(seq.find(Q(\'sheetData\')).iter(Q(\'c\'))):\n        if int(re.search(r\'\\d+\',c.get(\'r\')).group())>=4: set_cell(seq,c.get(\'r\'),None)\n    try:\n        for source_row,row in enumerate(w[\'Antibody Designs\'].iter_rows(min_row=4,max_col=37,values_only=True),4):\n            if not occupied(row[2]): continue\n            if any(occupied(v) for v in row[22:]):\n                raise ValueError(f\'Antibody Designs row {source_row}: bispecific/extra-construct fields cannot be represented in v3.1\')\n            count+=1; target=count+3\n            if target>1048576: raise ValueError(\'Too many designs for Excel\')\n            vals={dest:row[ord(src)-65] for src,dest in MAP.items()}\n            # Use source isotype when representable; retain sample-confirmed mini-binder mapping.\n            two=occupied(vals.get(\'S\')) or occupied(vals.get(\'M\'))\n            original_isotype=vals.get(\'D\')\n            if norm(original_isotype) == \'minibinder\':\n                vals[\'D\']=\'hIgG1\' if two else \'hIgG1 VHH Fc Fusion\'\n                report(f\'Antibody Designs!G{source_row}\',original_isotype,vals[\'D\'],\'mini-binder mapping matching supplied example; review\')\n            elif not occupied(original_isotype):\n                vals[\'D\']=\'hIgG1\' if two else \'hIgG1 VHH Fc Fusion\'\n                report(f\'Antibody Designs!G{source_row}\',None,vals[\'D\'],\'chain-based inference; review\')\n            if not two: vals[\'E\']=\'N/A\'\n            for col,cat in GROUPS.items():\n                v=vals.get(col)\n                if not occupied(v): continue\n                # Custom nucleotide sequences must never be fuzzy-replaced.\n                if col in (\'I\',\'K\',\'P\',\'R\') and re.fullmatch(r\'[ACGTUNacgtun\\s]+\',str(v)): continue\n                selected,status=closest(v,opts[cat])\n                if selected is None and col in (\'G\',\'N\'): selected=v; status=\'custom vector retained; review\'\n                vals[col]=selected\n                if status!=\'matched\': report(f\'Antibody Designs!{get_column_letter(ord(col)-64+3) if col!="C" else "C"}{source_row}\',v,selected,status)\n            for col in \'ABCDEFGHIJKLMNOPQRS\': set_cell(seq,f\'{col}{target}\',vals.get(col))\n        # Source field names, never row positions, drive Order Details mapping.\n        for r in range(3,18):\n            set_cell(details,f\'B{r}\',None); set_cell(details,f\'C{r}\',None)\n        labels={}\n        tw=load_workbook(TEMPLATE,read_only=True,data_only=True)\n        for r in range(3,18): labels[r]=[norm(tw[\'Order Details\'].cell(r,1).value)]+[norm(a) for a in ALIASES[r]]\n        tw.close(); seen=set()\n        source=next((w[n] for n in (\'Antibody Settings\',\'Order Details\') if n in w.sheetnames),None)\n        if source is None: report(\'Order Details\',None,None,\'source settings tab absent; all inputs left blank\')\n        else:\n            for cells in source.iter_rows(values_only=True):\n                if not cells or not occupied(cells[0]): continue\n                label=norm(cells[0])\n                if label in (\'antibodycustomizations\',\'orderdetailsrequired\',\'orderdetails\'): continue\n                matches=[r for r,aliases in labels.items() if label in aliases]\n                if not matches:\n                    ranked=sorted((max(difflib.SequenceMatcher(None,label,a).ratio() for a in aa),r) for r,aa in labels.items())\n                    if ranked[-1][0]>=.8 and ranked[-1][0]-ranked[-2][0]>=.08: matches=[ranked[-1][1]]\n                value=cells[1] if len(cells)>1 else None; note=cells[2] if len(cells)>2 else None\n                if not occupied(value) and not occupied(note): continue\n                if not matches:\n                    report(f\'{source.title}:{cells[0]}\',value,None,\'unmapped field; review\'); continue\n                r=matches[0]\n                if r in seen: report(f\'Order Details:{cells[0]}\',value,None,\'duplicate field ignored; review\'); continue\n                seen.add(r)\n                if occupied(value):\n                    if r in ORDER_CATEGORIES:\n                        selected,status=closest(value,opts[ORDER_CATEGORIES[r]])\n                        if status!=\'matched\': report(f\'Order Details!B{r}\',value,selected,status)\n                        set_cell(details,f\'B{r}\',selected)\n                        if selected is None: note=f\'{note or ""} [Original selection: {value}]\'.strip()\n                    else: set_cell(details,f\'B{r}\',value)\n                if occupied(note): set_cell(details,f\'C{r}\',note)\n        if \'Contact Information\' in w.sheetnames:\n            contact=roots[\'xl/worksheets/sheet2.xml\']\n            template_contact=load_workbook(TEMPLATE,read_only=True,data_only=True)\n            positions={}\n            for rr in range(4,30):\n                key=norm(template_contact[\'Contact Information\'].cell(rr,1).value)\n                if occupied(template_contact[\'Contact Information\'].cell(rr,1).value): positions.setdefault(key,[]).append(rr)\n            occurrence={}\n            for row in w[\'Contact Information\'].iter_rows(max_col=2,values_only=True):\n                if not occupied(row[0]): continue\n                label=norm(row[0]); index=occurrence.get(label,0)\n                if label in positions:\n                    occurrence[label]=index+1\n                    if index<len(positions[label]) and occupied(row[1]): set_cell(contact,f\'B{positions[label][index]}\',row[1])\n            template_contact.close()\n        # Refresh cached results for the template\'s SEARCH-based completion formulas.\n        lookup=load_workbook(TEMPLATE,read_only=True,data_only=True)\n        search_values=[str(r[0]).lower() for r in lookup[\'Sheet3\'].iter_rows(min_row=21,max_row=75,min_col=2,max_col=2,values_only=True) if occupied(r[0])]\n        lookup.close()\n        for c in details.iter(Q(\'c\')):\n            f=c.find(Q(\'f\'))\n            if f is not None and f.text and \'SEARCH(Sheet3!$B$21:$B$75,B\' in f.text:\n                address=\'B\'+re.search(r\'\\d+\',c.get(\'r\')).group()\n                target=details.find(f".//{Q(\'c\')}[@r=\'{address}\']")\n                value=\'\'.join(target.itertext()).lower() if target is not None else \'\'\n                cached=c.find(Q(\'v\'))\n                if cached is None: cached=ET.SubElement(c,Q(\'v\'))\n                cached.text=\'1\' if any(v in value for v in search_values) else \'0\'\n        dimension=seq.find(Q(\'dimension\'))\n        dimension.set(\'ref\',f\'A1:S{max(4568,count+3)}\')\n        # Remove standard and extended dropdown controls, retain other validation.\n        for root in roots.values():\n            for el in list(root.iter()):\n                if ET.QName(el).localname==\'dataValidation\' and el.get(\'type\')==\'list\': el.getparent().remove(el)\n            for el in list(root.iter()):\n                if ET.QName(el).localname==\'dataValidations\':\n                    if len(el)==0: el.getparent().remove(el)\n                    else: el.set(\'count\',str(len(el)))\n        out=io.BytesIO()\n        with zipfile.ZipFile(out,\'w\',zipfile.ZIP_DEFLATED) as dest:\n            for entry in z.infolist(): dest.writestr(entry,ET.tostring(roots[entry.filename],xml_declaration=True,encoding=\'UTF-8\',standalone=True) if entry.filename in roots else z.read(entry.filename))\n        return out.getvalue(),count,issues\n    finally: w.close(); z.close()\n\ndef run_batch(uploaded, output_dir=\'converted\'):\n    outdir=Path(output_dir); outdir.mkdir(parents=True,exist_ok=True); results=[]\n    for name,blob in uploaded.items():\n        if not name.lower().endswith(\'.xlsx\'): continue\n        stem=Path(name.replace(\'\\\\\',\'/\')).stem\n        content,count,issues=convert(blob)\n        base=f\'{stem}_converted_v{VERSION}\'\n        path=outdir/f\'{base}.xlsx\'; path.write_bytes(content)\n        report={\'input\':name,\'converter_version\':VERSION,\'design_count\':count,\'review_items\':issues}\n        report_path=outdir/f\'{base}_report.json\'; report_path.write_text(json.dumps(report,indent=2,ensure_ascii=False,default=str))\n        archive=outdir/f\'{base}.zip\'\n        with zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED) as z:\n            z.write(path,path.name); z.write(report_path,report_path.name)\n        results.append(archive)\n    return results\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(); p.add_argument(\'inputs\',nargs=\'+\',type=Path); p.add_argument(\'--output-dir\',default=\'converted\'); a=p.parse_args()\n    for path in run_batch({p.name:p.read_bytes() for p in a.inputs},a.output_dir): print(path)\n'
_namespace = {'__name__': 'converter_colab', '__file__': '/content/convert.py'}
exec(_module_source, _namespace)
_template_path = Path(tempfile.gettempdir()) / 'v3.1_converter_template.xlsx'
_template_path.write_bytes(_template_bytes)
_namespace['TEMPLATE'] = _template_path

"""Clean Colab entry point; converter namespace is supplied by the notebook bootstrap."""
import html
from pathlib import Path
from IPython.display import display, HTML
import ipywidgets as widgets
from google.colab import files


def uploaded_items(value):
    """Support both ipywidgets 7 (mapping) and 8 (tuple) payloads."""
    if isinstance(value, dict):
        return [(name, bytes(item['content'])) for name, item in value.items()]
    return [(item['name'], bytes(item['content'])) for item in value]


def show_converter(converter, output_dir='/content/converted'):
    display(HTML('''<div style="font-family:Arial,sans-serif;margin:8px 0 16px">
    <h2 style="margin:0 0 8px">Antibody form converter</h2>
    <div>v4.2 → v3.1 · Version 1.02</div>
    <p>Select one or more Excel files. Conversion starts automatically.</p></div>'''))
    picker=widgets.FileUpload(accept='.xlsx', multiple=True, description='Choose files',
        layout=widgets.Layout(width='240px',height='44px'),button_style='primary')
    results=widgets.VBox()

    def handle(change):
        items=uploaded_items(change['new'])
        if not items: return
        picker.disabled=True
        rows=[]
        try:
            for name,blob in items:
                status=widgets.HTML(value=f'Converting {html.escape(name)}…')
                rows.append(status); results.children=tuple(rows)
                try:
                    if not name.lower().endswith('.xlsx'):
                        raise ValueError('Please choose an .xlsx workbook.')
                    archives=converter({name:blob},output_dir)
                    archive=archives[0]
                    status.value=f'<b>✓ {html.escape(name)}</b><br>Ready. The ZIP includes your workbook and review report.'
                    download=widgets.Button(description='Download ZIP',button_style='success',
                        tooltip=archive.name,layout=widgets.Layout(width='180px'))
                    download.on_click(lambda _,p=str(archive): files.download(p))
                    rows.append(download)
                except Exception as exc:
                    status.value=f'<b>{html.escape(name)}</b><br>{html.escape(str(exc))}'
                results.children=tuple(rows)
        finally:
            picker.disabled=False
            # Reset so choosing the same files again triggers conversion.
            picker.unobserve(handle,names='value')
            picker.value={} if isinstance(picker.value,dict) else ()
            picker.observe(handle,names='value')
    picker.observe(handle,names='value')
    display(picker,results)
    return picker,results

clear_output(wait=True)
_converter_controls = show_converter(_namespace['run_batch'])
